In [4]:
# 시나리오 1 — 단계 함수들
def load_orders(path):
    # CSV에서 주문 데이터를 읽어 옵니다.
    return pd.read_csv(path)

def clean_strings_full(df):
    # 문자열 컬럼의 공백·대소문자·표기 혼재를 정리합니다.
    return df.assign(
        region=df["region"].str.strip().replace({"Seoul": "서울"}),
        membership=df["membership"].str.lower(),
        channel=df["channel"].str.strip().str.lower(),
    )

def drop_invalid_rows(df, age_min=10, age_max=80, qty_max=10):
    # 불가능한 값과 결측을 제거합니다.
    return (
        df
        .dropna(subset=["amount", "customer_age"])
        .query("@age_min < customer_age < @age_max")
        .query("quantity <= @qty_max")
        .drop_duplicates()
        .reset_index(drop=True)
    )

def add_derived(df):
    # 분석용 파생 컬럼을 추가합니다.
    return df.assign(
        amount_log=lambda d: np.log1p(d["amount"]),
        is_premium=lambda d: d["membership"].isin(["gold", "vip"]).astype(int),
        amount_class=lambda d: np.where(d["amount"] >= 100_000, "high",
                                np.where(d["amount"] >= 30_000, "mid", "low"))
    )

def encode_full(df):
    # membership=Ordinal, 나머지=One-Hot.
    order_map = {"basic": 1, "silver": 2, "gold": 3, "vip": 4}
    out = df.assign(membership_ord=df["membership"].map(order_map))
    one_hots = [
        pd.get_dummies(out["region"],   prefix="region", dtype=int),
        pd.get_dummies(out["channel"],  prefix="ch",     dtype=int),
        pd.get_dummies(out["category"], prefix="cat",    dtype=int),
    ]
    return pd.concat([out] + one_hots, axis=1)

def scale_with_robust(df, cols=("customer_age", "amount", "quantity")):
    # 수치형 컬럼을 Robust로 스케일링하고 _scaled 접미사를 붙입니다.
    scaler = RobustScaler()
    scaled = scaler.fit_transform(df[list(cols)])
    scaled_df = pd.DataFrame(scaled,
                             columns=[f"{c}_scaled" for c in cols],
                             index=df.index)
    return pd.concat([df, scaled_df], axis=1)

print("단계 함수 6개 정의 완료. 다음 셀에서 한 줄로 조립합니다.")

단계 함수 6개 정의 완료. 다음 셀에서 한 줄로 조립합니다.


In [ ]:
# 시나리오 2 — end-to-end 파이프라인
def preprocess(input_path):
    # 원본 CSV 경로를 받아 전처리된 DataFrame을 돌려줍니다.
    return (
        load_orders(input_path)
        .pipe(clean_strings_full)
        .pipe(drop_invalid_rows, age_min=10, age_max=80, qty_max=10)
        .pipe(add_derived)
        .pipe(encode_full)
        .pipe(scale_with_robust)
    )

# 진짜 한 줄 호출
clean_df = preprocess(input_csv)
print("입력 행 수 → 출력 행 수:", orders.shape[0], "→", clean_df.shape[0])
print("출력 컬럼 수:", clean_df.shape[1])
clean_df.head(3)

입력 행 수 → 출력 행 수: 1500 → 1404
출력 컬럼 수: 40
  order_id  customer_age  ... amount_scaled quantity_scaled
0   O00002          24.0  ...     10.141429             2.0
1   O00003          45.0  ...     -0.284286             0.0
2   O00005          41.0  ...     -0.284286             0.0

[3 rows x 40 columns]

In [ ]:
# 품질 리포트 함수
def quality_report(before: pd.DataFrame, after: pd.DataFrame) -> dict:
    # 전처리 전·후 데이터의 품질 차이를 dict로 반환합니다.
    report = {
        "rows_before": before.shape[0],
        "rows_after":  after.shape[0],
        "removed_pct": round(100 * (1 - after.shape[0] / before.shape[0]), 2),
        "cols_before": before.shape[1],
        "cols_after":  after.shape[1],
        "new_cols":    after.shape[1] - before.shape[1],
        #"missing_top5_before": (
        #    before.isnull().sum().sort_values(ascending=False).head(5).to_dict()
        #),
        #"missing_top5_after": (
        #   after.isnull().sum().sort_values(ascending=False).head(5).to_dict()
        #),
        "missing_top5_before": (
            (before.isnull().mean() * 100).round(2)
            .sort_values(ascending=False).head(5).to_dict()
        ),
        "missing_top5_after": (
            (after.isnull().mean() * 100).round(2)
            .sort_values(ascending=False).head(5).to_dict()
        ),
        "dtypes_after": after.dtypes.value_counts().to_dict(),
    }
    return report

# 사용
rep = quality_report(orders, clean_df)
for k, v in rep.items():
    print(f"{k}: {v}")

rows_before: 1500
rows_after: 1404
removed_pct: 6.4
cols_before: 22
cols_after: 40
new_cols: 18
#missing_top5_before: {'amount': 60, 'grade_apply': 60, 'grade_vec': 60, 'age_next_year_apply': 30, #'customer_age': 30}
#missing_top5_after: {'membership_score': 1, 'order_id': 0, 'region': 0, 'customer_age': 0, 'channel': #0}
missing_top5_before: {'amount': 4.0, 'grade_apply': 4.0, 'grade_vec': 4.0, 'age_next_year_apply': 2.0, 'customer_age': 2.0}
missing_top5_after: {'membership_score': 0.07, 'order_id': 0.0, 'region': 0.0, 'customer_age': 0.0, 'channel': 0.0}
dtypes_after: {dtype('int64'): 17, dtype('float64'): 12, dtype('O'): 11}


#개념퀴즈
1. 연산 속도와 유연성 비교apply, map, 벡터화 중 가장 빠르지만 가장 유연성이 떨어지는 것은?정답: (c) 벡터화이유: 판다스/넘파이의 벡터화 연산($s * 1.1$ 같은 것)은 내부적으로 C언어로 최적화되어 돌기 때문에 압도적으로 가장 빠릅니다. 하지만 딕셔너리 변환이나 복잡한 파이썬 커스텀 함수(if-else 조건문 등)를 적용하는 유연한 작업은 처리하지 못한다는 단점이 있습니다.

2. 범주형 인코딩의 선택다음 중 순서가 없는 카테고리에 가장 안전한 인코딩은?정답: (c) One-Hot이유: '서울, 부산, 대구'처럼 순서가 없는 카테고리에 숫자를 바로 매기면 모델이 수학적 서열로 오해합니다. 각 카테고리를 $0$과 $1$의 독립된 열로 쪼개주는 One-Hot 인코딩이 가장 안전합니다.

3. 이상치 대응 스케일러이상치가 있는 수치형 컬럼에 가장 안전한 스케일러는?정답: (c) Robust이유: MinMax는 이상치에 의해 전체 범위가 찌그러지고, Standard는 평균이 끌려 올라갑니다. 반면 Robust 스케일러는 중간값(Median)과 사분위수를 기준으로 삼기 때문에 이상치의 영향을 거의 받지 않아 가장 안전합니다.

4. 파이프라인의 내부 원리df.pipe(func)와 동등한 표현은?정답: (b) func(df)이유: .pipe(func)는 겉보기엔 특별해 보이지만, 실제로는 내 데이터프레임(df)을 내가 만든 함수(func)의 첫 번째 인자로 쏙 집어넣어 실행(func(df))해주는 편리한 포장지일 뿐입니다. 덕분에 코드를 꼬이지 않고 위에서 아래로 흐르게(Chaining) 쓸 수 있는 것이죠!

코드 퀴즈
orders에서 다음 작업을 한 줄짜리 chaining으로 작성하세요.

▸
amount 결측 제거
▸
region의 공백 제거(str.strip())
▸
customer_age가 10~80인 행만 남기기
▸
amount_log = log1p(amount) 컬럼 추가
▸
amount_log로 정렬 (큰 순)
▸
상위 5개만 보기

In [ ]:
# 코드 퀴즈 — 모범 답안
top5 = (
    orders
    .dropna(subset=["amount"])
    .assign(region=lambda d: d["region"].str.strip())
    .query("10 < customer_age < 80")
    .assign(amount_log=lambda d: np.log1p(d["amount"]))
    .sort_values("amount_log", ascending=False)
    .head(5)
)
print(top5[["region", "customer_age", "amount", "amount_log"]])

     region  customer_age    amount  amount_log
1        대구          24.0  749700.0    13.52743
658      경기          33.0  749700.0    13.52743
530      인천          49.0  749700.0    13.52743
1081     서울          36.0  749700.0    13.52743
95       경기          39.0  749700.0    13.52743